# CNN - 합성곱 신경망

In [26]:
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [27]:
(train_input, train_target), (test_input, test_target)= keras.datasets.fashion_mnist.load_data()

In [28]:
train_scaled = train_input.reshape(-1, 28, 28, 1) / 255.0
test_scaled = test_input.reshape(-1, 28, 28, 1) / 255.0

In [29]:
train_scaled.shape  # 4차원
# 합성곱 Conv2D는 입력값으로 4차원 텐서를 요구
# 채널이 1(흑백)이라서 28 x 28 x 1 이지만,
# 컬러는 reshape(-1, Height, width, 3) R, G, B 3채널
# 4차원 텐서 순서(batch, H, W, Channel)

(60000, 28, 28, 1)

In [30]:
model = keras.Sequential()

In [31]:
model.add(keras.layers.Input(shape=(28,28,1))) # 입력층 4차원 (배치) (h, w, channel)

In [32]:
# Layer 1
model.add(keras.layers.Conv2D(32, 3, activation='relu', padding='same')) # 필터 갯수를 32개, 3 x 3 필터 ReLU, 패딩 설정
# 출력 shape (28, 28, 32), 파라미터 수 : (3*3*1+1)*32 = 320
model.add(keras.layers.MaxPool2D(2))
# 출력 shape (14, 14, 32), 파라미터 수 : 0

In [33]:
# Layer 2
model.add(keras.layers.Conv2D(64, 3, activation='relu', padding='same')) # 필터 갯수를 64개, 3 x 3 필터 ReLU, 패딩 설정
# 출력 shape (14, 14, 64), 파라미터 수 : (3*3*1+1)*64 = 18496
model.add(keras.layers.MaxPool2D(2))
# 출력 shape (7, 7, 64), 파라미터 수 : 0

## 입력층과 출력층의 모양
Input shape: A 4D tensor with shape: (batch_size, height, width, channels)
Output shape: A 4D tensor with shape: (batch_size, new_height, new_width, filters)

# 다운 샘플링

연산량을 줄이고, 속도를 늘리고, 추상화를 개념화 시키기 위해, 해상도, 특징을 압축하고 요약하는 단계

In [34]:
# Full-Conneted Layer
model.add(keras.layers.Flatten())   # 출력 7 * 7 * 64 = 3136 (batch, 3136)
# flatten 층의 파라미터는 없음

In [35]:
model.add(keras.layers.Dense(100, activation='relu'))   # 파라미터 3136 * 100 + 100 = 313,700

In [36]:
model.add(keras.layers.Dropout(0.4)) # 과적합 피하기 위한 드롭아웃

In [37]:
# 출력층 : 구별하고자 하는 분류 갯수만큼 (class) Unit 설정
model.add(keras.layers.Dense(10, activation='softmax')) # 파라미터 100 * 10 + 10 = 1,010

In [38]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │       313,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 333,526 (1.27 MB)

 Trainable params: 333,526 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
model.compile(optimizer='adam', 
              loss=keras.losses.sparse_categorical_crossentropy,
              metrics=['accuracy'],
              jit_compile=False)

In [40]:
train_scaled.shape

(60000, 28, 28, 1)

In [41]:
# 콜백 설정 (checkpoint, early_stopping)
checkpoint = keras.callbacks.ModelCheckpoint('best-model.keras', save_best_only=True)
early_stopping = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)

In [42]:
model.fit(train_scaled, train_target, 
        epochs=20,
        callbacks=[checkpoint, early_stopping], 
        validation_split=0.2) # 48,000 : 12,000 (train, val)

Epoch 1/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8091 - loss: 0.5303 - val_accuracy: 0.8773 - val_loss: 0.3348
Epoch 2/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8731 - loss: 0.3510 - val_accuracy: 0.8971 - val_loss: 0.2816
Epoch 3/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.8904 - loss: 0.3028 - val_accuracy: 0.9000 - val_loss: 0.2721
Epoch 4/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9029 - loss: 0.2704 - val_accuracy: 0.9057 - val_loss: 0.2520
Epoch 5/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9087 - loss: 0.2468 - val_accuracy: 0.9133 - val_loss: 0.2341
Epoch 6/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9161 - loss: 0.2262 - val_accuracy: 0.9130 - val_loss: 0.2353
Epoch 7/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9232 - loss: 0.2105 - val_accuracy: 0.9123 - val_loss: 0.2446


In [43]:
import matplotlib.pyplot as plt
plt.plot(history.history['loss'])
plt.plot(history.history)

NameError: name 'history' is not defined

In [ ]:
# 검증 데이터 평가
model.evaluate()